## 연습 1: Pristine Properties 플랫폼 설정

데이터 작업을 시작하기 전에 Unity Catalog 구조를 구성하고 몇 가지 매물 목록을 로드해야 합니다. 이 연습의 모든 셀은 제공되어 있습니다 — 순서대로 실행하기만 하면 됩니다.

### ✅ 제공됨: Unity Catalog 구조 생성

아래 셀을 실행하여 다음을 생성합니다:
- `realestate_lab` 라는 카탈로그
- 원시 수집 데이터를 위한 **bronze** 스키마
- 정제 및 변환된 데이터를 위한 **silver** 스키마
- `realestate_lab.bronze` 내부의 관리형 볼륨 `raw_files`

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS realestate_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Pristine Properties — 네덜란드 도시를 위한 부동산 분석 플랫폼'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS realestate_lab.bronze
  COMMENT '원시 수집 데이터 — 중개 사무소에서 받은 그대로';

CREATE SCHEMA IF NOT EXISTS realestate_lab.silver
  COMMENT '정제 및 변환된 데이터 — 분석 준비 완료';

CREATE VOLUME IF NOT EXISTS realestate_lab.bronze.raw_files
  COMMENT '중개 시스템에서 받은 원시 CSV 파일의 랜딩 존';

### ✅ 제공됨: 샘플 데이터로 볼륨 채우기

아래 셀을 실행하여 네 개의 샘플 CSV 파일을 볼륨에 기록합니다:
- `property_listings.csv` — 의도적인 품질 문제(중복 및 null)가 포함된 33개의 매물 목록
- `real_estate_agents.csv` — 네덜란드 7개 도시에 걸친 8명의 중개인 프로필
- `property_sales.csv` — 완료된 3건의 판매 거래
- `market_stats_wide.csv` — 도시별 월간 평균 매물 가격(와이드 형식)

이 파일들은 연습 2–6 전반에 걸쳐 사용합니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 샘플 CSV 파일을 볼륨에 기록합니다

listings_csv = """listing_id,property_id,agent_id,city,property_type,bedrooms,bathrooms,list_price,sqm,listing_date,last_updated,status
1,PROP-001,101,Amsterdam,Apartment,2,1,285000.50,75,2026-01-15,2026-01-15 09:00:00,Active
2,PROP-002,102,Rotterdam,House,4,2,449999.99,145,2026-01-20,2026-01-20 10:30:00,Active
3,PROP-003,101,Amsterdam,Studio,,1,184999.99,42,2026-01-22,2026-01-22 11:00:00,Active
4,PROP-004,103,Utrecht,Villa,5,3,874999.99,280,2026-02-01,2026-02-01 08:45:00,Active
5,PROP-005,102,Rotterdam,Apartment,3,2,320500.25,95,2026-02-05,2026-02-05 14:00:00,Sold
6,PROP-001,101,Amsterdam,Apartment,2,1,279000.00,75,2026-01-15,2026-03-01 09:00:00,Active
7,PROP-006,104,The Hague,House,3,2,395000.00,120,2026-02-10,2026-02-10 16:00:00,Active
8,PROP-007,103,Utrecht,Apartment,1,,165000.00,38,2026-02-14,2026-02-14 11:30:00,Active
9,PROP-008,105,Eindhoven,House,4,3,519999.99,170,2026-02-18,2026-02-18 09:00:00,Active
10,PROP-009,104,The Hague,Studio,1,1,145000.00,35,2026-02-22,2026-02-22 10:00:00,Active
11,PROP-010,101,Amsterdam,Villa,6,4,1249999.99,400,2026-03-01,2026-03-01 14:30:00,Active
12,PROP-002,102,Rotterdam,House,4,2,445000.00,145,2026-01-20,2026-03-05 10:30:00,Active
13,PROP-011,105,Eindhoven,Apartment,2,1,195000.00,68,2026-03-08,2026-03-08 09:00:00,Sold
14,PROP-012,106,Groningen,House,3,,309999.99,105,2026-03-10,2026-03-10 12:00:00,Active
15,PROP-013,102,Rotterdam,Apartment,2,2,265000.00,82,2026-03-15,2026-03-15 15:00:00,Active
16,PROP-014,106,Groningen,Studio,,1,129999.99,33,2026-03-18,2026-03-18 08:30:00,Active
17,PROP-015,103,Utrecht,House,5,3,679999.99,210,2026-03-22,2026-03-22 11:00:00,Active
18,PROP-016,104,The Hague,Apartment,1,1,175000.00,45,2026-03-25,2026-03-25 14:00:00,Sold
19,PROP-017,101,Amsterdam,House,4,2,785000.50,195,2026-04-01,2026-04-01 09:30:00,Active
20,PROP-018,105,Eindhoven,Villa,4,3,649999.99,230,2026-04-05,2026-04-05 10:00:00,Active
21,PROP-019,106,Groningen,Apartment,3,2,215000.00,78,2026-04-08,2026-04-08 11:30:00,Active
22,PROP-020,107,Breda,House,3,2,355000.00,115,2026-04-12,2026-04-12 14:00:00,Active
23,PROP-003,101,Amsterdam,Studio,,1,190000.00,42,2026-01-22,2026-04-15 11:00:00,Active
24,PROP-021,107,Breda,Apartment,2,1,225000.00,72,2026-04-18,2026-04-18 09:00:00,Active
25,PROP-022,102,Rotterdam,Villa,5,,925000.00,310,2026-04-22,2026-04-22 16:00:00,Active
26,PROP-023,103,Utrecht,House,3,2,,105,2026-04-25,2026-04-25 10:30:00,
27,PROP-024,104,The Hague,Apartment,2,1,235000.00,71,2026-05-01,2026-05-01 09:00:00,Active
28,PROP-025,105,Eindhoven,House,,2,410000.00,130,2026-05-05,2026-05-05 11:00:00,Active
29,PROP-026,106,Groningen,Studio,1,1,138000.00,36,2026-05-08,2026-05-08 14:00:00,Active
30,PROP-027,107,Breda,House,4,3,490000.00,158,2026-05-12,2026-05-12 09:30:00,Active
31,PROP-028,101,Amsterdam,Apartment,3,2,345000.00,98,2026-05-15,2026-05-15 10:00:00,Active
32,PROP-029,104,The Hague,House,4,2,,145,2026-05-18,2026-05-18 12:00:00,Active
33,PROP-030,103,Utrecht,Apartment,2,1,245000.00,76,2026-05-22,2026-05-22 15:00:00,Active"""

agents_csv = """agent_id,agent_name,agency,city,phone
101,Sophie van den Berg,Makelaardij De Kust,Amsterdam,+31-20-1234567
102,Jan de Vries,Vastgoed Noord,Rotterdam,+31-10-9876543
103,Maria Hendriks,De Hoek Makelaars,Utrecht,+31-30-4567890
104,Thomas Bakker,Haagse Woning,The Hague,+31-70-2345678
105,Lisa Smits,Brainport Vastgoed,Eindhoven,+31-40-3456789
106,Pieter Jansen,Stadspark Makelaars,Groningen,+31-50-8765432
107,Emma Koster,Markdal Vastgoed,Breda,+31-76-5678901
108,Robert Mulder,Canal Ring Realty,Amsterdam,+31-20-9012345"""

sales_csv = """sale_id,property_id,sale_price,sale_date,days_on_market
1,PROP-005,315000.00,2026-03-15,38
2,PROP-011,192000.00,2026-04-20,43
3,PROP-016,172500.00,2026-04-30,35"""

market_csv = """city,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Amsterdam,385000,392000,401000,415000,422000,438000,445000,441000,428000,418000,405000,398000
Rotterdam,295000,298000,305000,312000,318000,325000,321000,315000,308000,301000,295000,290000
Utrecht,310000,315000,322000,328000,335000,342000,348000,344000,336000,328000,318000,312000
The Hague,275000,280000,288000,295000,302000,308000,312000,306000,298000,291000,283000,278000
Eindhoven,258000,262000,268000,275000,282000,288000,293000,289000,281000,273000,265000,260000
Groningen,195000,198000,203000,208000,213000,218000,222000,219000,214000,208000,201000,196000
Breda,245000,249000,255000,262000,268000,274000,279000,275000,268000,260000,253000,247000"""

base = "/Volumes/realestate_lab/bronze/raw_files"
dbutils.fs.put(f"{base}/property_listings.csv",   listings_csv, overwrite=True)
dbutils.fs.put(f"{base}/real_estate_agents.csv",  agents_csv,   overwrite=True)
dbutils.fs.put(f"{base}/property_sales.csv",      sales_csv,    overwrite=True)
dbutils.fs.put(f"{base}/market_stats_wide.csv",   market_csv,   overwrite=True)
print("CSV 파일이 볼륨에 기록되었습니다.")

### ✅ 제공됨: bronze Delta 테이블로 데이터 로드

아래 셀을 실행하여 볼륨에서 네 개의 CSV 파일을 모두 읽고 `realestate_lab.bronze` 스키마에 Delta 테이블로 저장합니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — CSV 파일을 bronze Delta 테이블로 로드합니다

def load_csv(path, table):
    (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
        .write.mode("overwrite")
        .saveAsTable(table))

base = "/Volumes/realestate_lab/bronze/raw_files"
load_csv(f"{base}/property_listings.csv",  "realestate_lab.bronze.listings")
load_csv(f"{base}/real_estate_agents.csv", "realestate_lab.bronze.agents")
load_csv(f"{base}/property_sales.csv",     "realestate_lab.bronze.sales")
load_csv(f"{base}/market_stats_wide.csv",  "realestate_lab.bronze.market_stats")
print("모든 테이블이 성공적으로 로드되었습니다.")

## 연습 2: 매물 목록 데이터 프로파일링

데이터를 변환하기 전에 그 안에 무엇이 있는지 이해해야 합니다. 이 연습에서는 `listings` 테이블에 대한 통계를 계산하고 SQL과 PySpark를 모두 사용하여 요약 프로파일을 생성합니다.

> 💬 **노트북 외 작업:** 이 연습을 완료한 후, 랩 설정 페이지의 단계를 따라 Catalog Explorer에서 `realestate_lab.bronze.listings`에 대한 시각적 **스냅샷(Snapshot)** 프로파일을 생성하세요.

### 작업 1: 테이블 통계 계산

`ANALYZE TABLE`을 사용하여 `realestate_lab.bronze.listings`의 모든 열에 대한 통계를 계산하세요. 그런 다음 `DESCRIBE TABLE EXTENDED`를 사용하여 수집된 메트릭을 확인하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 `ANALYZE TABLE`이 어떤 통계를 수집하며 쿼리 옵티마이저에 어떻게 도움이 되는지 물어보세요.
> *추천 프롬프트: "What statistics does ANALYZE TABLE collect in Databricks and how does the query optimizer use them?"*

In [ ]:
%sql
-- 모든 열에 대한 통계 계산
ANALYZE TABLE realestate_lab.bronze.listings COMPUTE STATISTICS FOR ALL COLUMNS;

In [ ]:
%sql
-- 수집된 통계 확인
DESCRIBE TABLE EXTENDED realestate_lab.bronze.listings;

### 작업 2: PySpark를 사용하여 요약 통계 생성

`realestate_lab.bronze.listings` 테이블을 PySpark DataFrame으로 로드하고 `.describe()`를 사용하여 모든 열의 요약 통계를 확인하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 `.describe()`가 무엇을 반환하며 `.summary()`와 어떻게 다른지 물어보세요.
> *추천 프롬프트: "What is the difference between df.describe() and df.summary() in PySpark?"*

In [ ]:
df = spark.table("realestate_lab.bronze.listings")
df.describe().show()

## 연습 3: 올바른 데이터 타입 선택

올바른 데이터 타입을 선택하면 데이터의 정확성, 정밀도, 성능에 영향을 줍니다. `DECIMAL` 대 `DOUBLE`, `TIMESTAMP` 대 `TIMESTAMP_NTZ`, 그리고 복합 타입인 `ARRAY`, `MAP`, `STRUCT`라는 세 가지 핵심 결정을 살펴봅니다.

### 작업 1: 부동산 가격을 위한 DECIMAL 대 DOUBLE

`DOUBLE` 같은 부동소수점 타입은 내부적으로 이진 표현을 사용하므로 작지만 중요한 반올림 오류가 발생할 수 있습니다 — 부동산 가격과 같은 금융 데이터에서는 심각한 문제입니다.

먼저, 제공된 셀을 실행하여 이 문제를 관찰하세요. 그런 다음 bronze 테이블로부터 `realestate_lab.silver.listings_typed`를 생성하여 `list_price`를 `DECIMAL(12,2)`로 저장하는 `CREATE TABLE` 문을 작성하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 왜 `DOUBLE`이 가격에서 정밀도 오류를 일으키는지 물어보세요.
> *추천 프롬프트: "Why does storing a price like 449999.99 as DOUBLE in SQL lead to precision errors, and what type should I use instead?"*

In [ ]:
%sql
-- ✅ 제공됨: 실행하여 DOUBLE의 부동소수점 부정확성을 관찰하세요
SELECT
  -- DOUBLE에서 0.1 + 0.2 = 0.3 일까요?
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) as sum_double,
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) = CAST(0.3 AS DOUBLE)        AS double_exact,   -- FALSE 반환
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) as sum_decimal,
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) = CAST(0.3 AS DECIMAL(2,1))        AS decimal_exact  -- TRUE 반환

In [ ]:
%sql
-- 올바르게 타입이 지정된 silver 테이블 생성 -- list_price에 DECIMAL(12,2) 주목
CREATE OR REPLACE TABLE realestate_lab.silver.listings_typed AS
SELECT
    CAST(listing_id    AS INT)           AS listing_id,
    property_id,
    CAST(agent_id      AS INT)           AS agent_id,
    city,
    property_type,
    CAST(bedrooms      AS INT)           AS bedrooms,
    CAST(bathrooms     AS DOUBLE)        AS bathrooms,
    CAST(list_price    AS DECIMAL(12,2)) AS list_price,
    CAST(sqm           AS INT)           AS sqm,
    CAST(listing_date  AS DATE)          AS listing_date,
    CAST(last_updated  AS TIMESTAMP)     AS last_updated,
    status
FROM realestate_lab.bronze.listings;

SELECT * FROM realestate_lab.silver.listings_typed LIMIT 5;

### 작업 2: 매물 관람 예약을 위한 TIMESTAMP 대 TIMESTAMP_NTZ

부동산 관람 일정을 잡을 때는 올바른 시간 타입을 선택하세요:
- `TIMESTAMP`: UTC로 저장되며 세션 시간대에 맞춰 조정되어 표시됩니다. 전역 이벤트에 사용하세요.
- `TIMESTAMP_NTZ`: 입력한 그대로 저장되며 변환이 없습니다. 로컬에서 예약된 약속에 사용하세요.

제공된 셀을 실행하여 세션 시간대가 변경될 때 두 타입이 어떻게 다르게 동작하는지 확인하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 언제 `TIMESTAMP`보다 `TIMESTAMP_NTZ`를 선호해야 하는지 물어보세요.
> *추천 프롬프트: "When should I use TIMESTAMP_NTZ instead of TIMESTAMP in Azure Databricks for property viewing appointments?"*

In [ ]:
%sql
-- ✅ 제공됨: TIMESTAMP와 TIMESTAMP_NTZ가 시간대 변경에 어떻게 반응하는지 관찰하세요
CREATE OR REPLACE TEMP VIEW viewing_appointments AS
SELECT *
FROM VALUES
  ('VA001', 'PROP-001', TIMESTAMP '2026-06-15 09:00:00', TIMESTAMP_NTZ '2026-06-15 09:00:00'),
  ('VA002', 'PROP-005', TIMESTAMP '2026-06-15 14:30:00', TIMESTAMP_NTZ '2026-06-15 14:30:00'),
  ('VA003', 'PROP-010', TIMESTAMP '2026-06-16 10:00:00', TIMESTAMP_NTZ '2026-06-16 10:00:00')
AS t(appointment_id, property_id, viewing_time_ts, viewing_time_ntz);

-- 뉴욕 시간대로 전환 -- TIMESTAMP 값은 이동하고; TIMESTAMP_NTZ는 변경되지 않습니다
SET TIME ZONE 'America/New_York';
SELECT appointment_id, property_id, viewing_time_ts, viewing_time_ntz
FROM viewing_appointments;

In [ ]:
%sql
-- ✅ 완료되면 세션 시간대를 다시 UTC로 재설정하세요
SET TIME ZONE 'UTC';

### 작업 3: 복합 타입 — ARRAY, MAP, STRUCT

매물 목록은 종종 중첩되거나 반복되는 데이터를 포함합니다: 편의시설 목록, 유연한 특성 집합, 구조화된 주소 등입니다. 이는 자연스럽게 `ARRAY`, `MAP`, `STRUCT`로 매핑됩니다.

**작업 3a:** 다음 열을 가진 `listings_enriched`라는 임시 뷰를 생성하세요 (`VALUES`를 사용하여 최소 3개 행의 샘플 데이터 제공):
- `property_id` STRING
- `amenities` ARRAY\<STRING\> — 예: `ARRAY('Parking', 'Balcony', 'Storage')`
- `features` MAP\<STRING, STRING\> — 예: `MAP('energy_label', 'A', 'heating', 'central')`
- `address` STRUCT\<street: STRING, city: STRING, postal_code: STRING\>

**작업 3b:** `listings_enriched`를 쿼리하여 다음에 접근하세요: 첫 번째 편의시설(인덱스 0), features의 `energy_label`, address의 `city`.

> 🤖 **Genie Code 팁:** Databricks SQL에서 ARRAY, MAP, STRUCT를 정의하고 접근하는 방법을 물어보세요.
> *추천 프롬프트: "How do I create a temp view in Databricks SQL with ARRAY, MAP, and STRUCT columns, and how do I access elements from each?"*

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW listings_enriched AS
SELECT *
FROM VALUES
  ('PROP-001',
   ARRAY('Parking', 'Balcony', 'Storage'),
   MAP('energy_label', 'B', 'heating', 'central'),
   STRUCT('Keizersgracht 45' AS street, 'Amsterdam' AS city, '1017 DK' AS postal_code)),
  ('PROP-002',
   ARRAY('Garden', 'Garage', 'Shed'),
   MAP('energy_label', 'C', 'heating', 'gas'),
   STRUCT('Blaak 22' AS street, 'Rotterdam' AS city, '3011 TA' AS postal_code)),
  ('PROP-004',
   ARRAY('Pool', 'Sauna', 'Home Cinema', 'Parking'),
   MAP('energy_label', 'A', 'heating', 'heat_pump'),
   STRUCT('Wilhelminapark 8' AS street, 'Utrecht' AS city, '3581 MK' AS postal_code))
AS t(property_id, amenities, features, address);

SELECT * FROM listings_enriched;

In [ ]:
%sql
-- 각 복합 타입에서 요소 접근
SELECT
    property_id,
    amenities[0]              AS primary_amenity,
    features['energy_label']  AS energy_label,
    address.city              AS address_city
FROM listings_enriched;

## 연습 4: 중복 및 누락 값 처리

원시 매물 목록 데이터에는 중복 레코드(가격이 갱신되어 다시 제출된 동일 매물)와 여러 열의 누락 값이 포함되어 있습니다. 이 연습에서는 두 문제를 모두 식별하고 해결합니다.

### 작업 1: 중복된 매물 목록 식별

`GROUP BY`와 `HAVING`을 사용하여 두 번 이상 나타나는 모든 `property_id` 값을 찾는 SQL 쿼리를 작성하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 중복 값을 찾는 SQL 패턴을 물어보세요.
> *추천 프롬프트: "Write a SQL query to find duplicate property_id values in realestate_lab.bronze.listings."*

In [ ]:
%sql
-- 두 번 이상 나타나는 property_id 값 찾기
SELECT
    property_id,
    COUNT(*) AS occurrence_count
FROM realestate_lab.bronze.listings
GROUP BY property_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

### 작업 2: 매물별 가장 최근에 갱신된 목록 유지

`ROW_NUMBER()`와 `QUALIFY` 절을 사용하여 각 `property_id`에 대해 가장 최근에 갱신된 행만 유지하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 중복 제거를 위한 `QUALIFY`를 사용한 `ROW_NUMBER()`에 대해 물어보세요.
> *추천 프롬프트: "How do I use ROW_NUMBER() with QUALIFY in Databricks SQL to keep only the latest record per property_id ordered by last_updated?"*

In [ ]:
%sql
-- 매물별로 가장 최근에 갱신된 행만 유지
SELECT *
FROM realestate_lab.bronze.listings
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY property_id
    ORDER BY last_updated DESC
) = 1
ORDER BY property_id;

### 작업 3: PySpark를 사용한 중복 제거

PySpark를 사용하여 listings 테이블을 로드하고, `last_updated` 내림차순으로 정렬한 다음 `dropDuplicates(["property_id"])`를 호출하세요. 전후의 개수를 출력하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 정렬이 `dropDuplicates`와 어떻게 상호작용하는지 물어보세요.
> *추천 프롬프트: "How does orderBy work together with dropDuplicates in PySpark to control which duplicate row is kept?"*

In [ ]:
from pyspark.sql.functions import col

df = spark.table("realestate_lab.bronze.listings")
print(f"중복 제거 전 개수: {df.count()}")

df_deduped = (
    df
    .orderBy(col("last_updated").desc())
    .dropDuplicates(["property_id"])
)

print(f"중복 제거 후 개수:  {df_deduped.count()}")
display(df_deduped)

### 작업 4: null 및 누락 값 처리

listings 테이블에는 여러 열에 null이 있습니다. 다음 규칙을 적용하세요:
- `list_price`는 철회된 목록에서 null입니다 — **이 행들을 삭제**
- `bedrooms`는 일부 유닛에서 null입니다 — **`0`으로 채우기**
- `bathrooms`는 일부 매물에서 null입니다 — **`1.0`으로 채우기**
- `status`는 불완전한 레코드에서 null입니다 — **`'Unknown'`으로 채우기**

`dropna()`와 `fillna()`를 사용하세요. 전후 개수를 출력하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 PySpark에서 dropna와 fillna를 연결하는 방법을 물어보세요.
> *추천 프롬프트: "How do I use dropna and fillna together in PySpark to drop rows where list_price is null and then fill other null columns?"*

In [ ]:
df = spark.table("realestate_lab.bronze.listings")

# list_price가 null인 행 삭제 (철회된 목록)
df_clean = df.dropna(subset=["list_price"])

# 남은 null을 합리적인 기본값으로 채우기
df_clean = df_clean.fillna({
    "bedrooms": 0,
    "bathrooms": 1.0,
    "status": "Unknown"
})

print(f"원본 개수:  {df.count()}")
print(f"정제 후 개수:   {df_clean.count()}")
display(df_clean)

## 연습 5: 중개인 및 판매 데이터와 매물 목록 조인

이 연습에서는 PySpark 조인을 사용하여 listings 테이블을 중개인 프로필 및 판매 레코드와 결합합니다.

### 작업 1: listings와 agents의 inner join

`agent_id`를 기준으로 **inner join**을 사용하여 `realestate_lab.bronze.listings`를 `realestate_lab.bronze.agents`와 조인하세요. 다음을 선택하세요: `listing_id`, `property_id`, `city`(listings에서), `list_price`, `agent_name`, `agency`.

> 🤖 **Genie Code 팁:** Genie Code에게 열 별칭을 사용한 올바른 PySpark 조인 구문을 물어보세요.
> *추천 프롬프트: "How do I inner join two PySpark DataFrames on agent_id and select columns from both using DataFrame aliases?"*

In [ ]:
listings_df = spark.table("realestate_lab.bronze.listings")
agents_df   = spark.table("realestate_lab.bronze.agents")

result = (
    listings_df.alias("l")
    .join(agents_df.alias("a"), on="agent_id", how="inner")
    .select("l.listing_id", "l.property_id", "l.city", "l.list_price",
            "a.agent_name", "a.agency")
)

display(result)

### 작업 2: 목록이 없는 중개인 찾기

`agents`를 왼쪽에, `listings`를 오른쪽에 두고 **left join**을 수행하세요. `listing_id`가 null인 행을 필터링하여 현재 목록이 없는 중개인을 식별하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 left join이 한 테이블에는 존재하지만 다른 테이블에는 없는 레코드를 찾는 데 어떻게 도움이 되는지 물어보세요.
> *추천 프롬프트: "How do I use a left join in PySpark to find rows in agents that have no match in listings?"*

In [ ]:
listings_df = spark.table("realestate_lab.bronze.listings")
agents_df   = spark.table("realestate_lab.bronze.agents")

# agents를 왼쪽에 두고 left join하여 목록이 없는 중개인 찾기
result = (
    agents_df.alias("a")
    .join(listings_df.alias("l"), on="agent_id", how="left")
    .filter("l.listing_id IS NULL")
    .select("a.agent_id", "a.agent_name", "a.agency")
)

display(result)

### 작업 3: 판매되지 않은 목록 찾기

`property_id`를 기준으로 **left join**을 사용하여 `listings`(왼쪽)를 `sales`(오른쪽)와 조인하세요. `sale_id IS NULL`인 행을 필터링하여 한 번도 판매되지 않은 매물을 찾으세요.

> 🤖 **Genie Code 팁:** Genie Code에게 left join + null 필터 패턴에 대해 물어보세요.
> *추천 프롬프트: "How do I find rows in the listings table with no matching sale in the sales table using a PySpark left join?"*

In [ ]:
from pyspark.sql.functions import col

listings_df = spark.table("realestate_lab.bronze.listings")
sales_df    = spark.table("realestate_lab.bronze.sales")

result = (
    listings_df.alias("l")
    .join(sales_df.alias("s"), on="property_id", how="left")
    .filter(col("s.sale_id").isNull())
    .select("l.listing_id", "l.property_id", "l.city",
            "l.property_type", "l.list_price", "l.listing_date")
)

display(result)

## 연습 6: 시장 통계 피벗 및 언피벗

데이터는 종종 분석에 필요한 형태와 맞지 않는 모양으로 도착합니다. 이 연습에서는 행을 열로 회전(피벗)하고 와이드 형식의 열을 다시 행으로 변환(언피벗)합니다.

### 작업 1: 매물 유형별 목록 개수 피벗

bronze listings를 사용하여 도시별 **활성(active)** 목록 개수를 세는 SQL `PIVOT` 쿼리를 작성하되, `property_type`(`Apartment`, `House`, `Studio`, `Villa`)별로 하나의 열을 만드세요.

예상 출력 형태:

| city | Apartment | House | Studio | Villa |
|---|---|---|---|---|
| Amsterdam | 3 | 1 | 1 | 1 |

> 🤖 **Genie Code 팁:** Genie Code에게 Databricks SQL의 `PIVOT` 절 구문을 물어보세요.
> *추천 프롬프트: "How do I use the PIVOT clause in Databricks SQL to count active listings per city, with one column per property type?"*

In [ ]:
%sql
-- Active 목록에 대해 property_type별 목록 개수 피벗
SELECT city, Apartment, House, Studio, Villa
FROM (
    SELECT city, property_type
    FROM realestate_lab.bronze.listings
    WHERE status = 'Active'
)
PIVOT (
    COUNT(*)
    FOR property_type IN ('Apartment' AS Apartment, 'House' AS House,
                          'Studio' AS Studio, 'Villa' AS Villa)
)
ORDER BY city;

### 작업 2: 월간 시장 통계 언피벗

`realestate_lab.bronze.market_stats` 테이블은 도시당 한 행과 12개의 월간 가격 열을 가지고 있습니다. `UNPIVOT` 절을 사용하여 이를 `city`, `month`, `avg_price` 열을 가진 long 형식으로 변환하세요.

> 🤖 **Genie Code 팁:** Genie Code에게 Databricks SQL의 UNPIVOT 구문을 물어보세요.
> *추천 프롬프트: "How do I unpivot 12 monthly columns (Jan through Dec) into rows using the UNPIVOT clause in Databricks SQL?"*

In [ ]:
%sql
-- 12개의 월간 열을 long 형식으로 언피벗
SELECT *
FROM realestate_lab.bronze.market_stats
UNPIVOT (
    avg_price FOR month IN (
        Jan, Feb, Mar, Apr, May, Jun,
        Jul, Aug, Sep, Oct, Nov, Dec
    )
)
ORDER BY city, month;

## 정리

완료되면 아래 셀을 실행하여 모든 랩 리소스를 제거하세요.

In [ ]:
%sql
-- 완료되면 주석을 해제하고 실행하여 모든 랩 리소스를 제거하세요
-- DROP CATALOG IF EXISTS realestate_lab CASCADE;